# 02. Huấn luyện & Ablation Study cho BiLSTM-CRF (Kaggle / Colab)

Notebook của **Tín**: thiết lập môi trường trên Kaggle/Colab, tải dữ liệu `PhoNER_COVID19`, chạy mô hình `BiLSTM-CRF` đầy đủ (`CharCNN + BiLSTM + LinearChainCRF`) và các nhánh đối chứng (Ablation Study).

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/TTins1702/NER-PhoNER_COVID19-UET.git"
BRANCH = "tin-bilstm-crf"
REPO_DIR = "NER-PhoNER_COVID19-UET"
if Path("../src").exists():
    %cd ..
if not Path("src").exists():
    if not Path(REPO_DIR).exists():
        !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_DIR}
else:
    !git pull

!pip install -q -r requirements.txt
!python -m src.data.download --output-dir data

# Tự động lấy WANDB_API_KEY từ Colab Secrets / Kaggle Secrets; nếu không có thì chạy offline
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if os.environ.get("WANDB_API_KEY"):
    os.environ.pop("WANDB_MODE", None)
    print("WANDB_API_KEY loaded -> logging online to Weights & Biases")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_API_KEY not set -> running in WANDB_MODE=offline")

In [ ]:
# 1. Mô hình đầy đủ: BiLSTM + CharCNN + CRF
!python scripts/train.py --config configs/bilstm_crf.yaml

In [ ]:
# 2. Ablation 1: Tắt tầng CRF (BiLSTM + CharCNN + Softmax)
!python scripts/train.py --config configs/bilstm_crf.yaml --override experiment_name=bilstm_softmax_word model.use_crf=false training.checkpoint_dir=checkpoints/bilstm_softmax

# 3. Ablation 2: Tắt tầng Character Embedding (BiLSTM + CRF)
!python scripts/train.py --config configs/bilstm_crf.yaml --override experiment_name=bilstm_crf_nochar_word model.use_char=false training.checkpoint_dir=checkpoints/bilstm_crf_nochar

# 4. Ablation 3: Thay CharCNN bằng CharBiLSTM
!python scripts/train.py --config configs/bilstm_crf.yaml --override experiment_name=bilstm_charlstm_crf_word model.char_encoder_type=bilstm training.checkpoint_dir=checkpoints/bilstm_charlstm_crf

In [ ]:
import json

for path in sorted(Path("outputs/metrics").glob("bilstm*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    t = data["test"]
    print(
        f"{data['experiment_name']:28s} | P={t['micro_precision']:.4f} | "
        f"R={t['micro_recall']:.4f} | micro-F1={t['micro_f1']:.4f} | "
        f"macro-F1={t['macro_f1']:.4f} | BIO_viol={t['bio_violations']}"
    )